In [ ]:
#=================================================
# Cellule "0" Notebook LLM Visual Explorer (do NOT run)
#=================================================
'''
          LLM Visual Explorer V0.9.1

Goal:
-----
Explore how a language model organizes concepts
in a semantic vector space.

Pipeline:
---------

Scenario      (Cell 3)  Load or enter scenario (default from config.py)
    ↓
Embeddings    (Cell 4)  Compute embedding vectors
    ↓
Projection    (Cell 5)  Project embeddings into a 3D PCA space
    ↓
DataFrame     (Cell 6)  Build the dataframe for visualization
    ↓
Similarity    (Cell 7)  Compute cosine similarities
    ↓
Visualization (Cell 8)  Display the interactive 3D scene
    ↓
Report        (Cell 9)  Display the exploration report
'''

'\n          LLM Visual Explorer V0.9.1\n\nGoal:\n-----\nExplore how a language model organizes concepts\nin a semantic vector space.\n\nPipeline:\n---------\n\nScenario     (Cell 3)  Load or enter scenario (default from config.py)\n    ↓\nEmbeddings   (Cell 4)  Compute embedding vectors\n    ↓\nProjection   (Cell 5)  Project embeddings into a 3D PCA space\n    ↓\nSimilarity   (Cell 6)  Compute cosine similarities\n    ↓\nDataFrame    (Cell 7)  Build the dataframe for visualization\n    ↓\nVisualization(Cell 8)  Display the interactive 3D scene \n    ↓\nVisualization(Cell 9)  Display the exploration report \n'

In [36]:
# ==========================================================
# Cellule 1 - Initialization / environment
# ==========================================================

# Install required libraries
!pip -q install sentence-transformers plotly scikit-learn

In [37]:
#=================================================
# Cellule 2 - Initialization / imports
#=================================================

# Clone the project (only if not already present)
import os

if not os.path.exists("/content/LLM-Visual-Explorer"):
    !git clone https://github.com/TintinDeBrest/LLM-Visual-Explorer.git

# Make the package visible to Python
import sys

PROJECT_DIR = "/content/LLM-Visual-Explorer"

if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)


In [38]:
#=================================================
# Cellule 3 - Scenario
#=================================================

from explorer.config import DEFAULT_SCENARIO
from explorer.config import MODEL_NAME
from explorer.scenarios import load_scenario
from explorer.display import display_scenario

# Choix du scénario
SCENARIO_NAME = "animals" # Test du 01Aug26

scenario = load_scenario(SCENARIO_NAME)

display_scenario(scenario)

words = [
    obj["name"]
    for obj in scenario["objects"]
]

categories = [
    obj["category"]
    for obj in scenario["objects"]
]

SCÉNARIO : Animaux

Comparaison sémantique de cinq mammifères.

Objets analysés :

   • Chat
   • Lion
   • Chien
   • Loup
   • Renard

Nombre d'objets : 5


In [39]:
# ==========================================================
# Cellule 4 Chargement du modèle d'embeddings
# ==========================================================

from sentence_transformers import SentenceTransformer
from explorer.embeddings import compute_embeddings

# Chargement du modèle
model = SentenceTransformer(MODEL_NAME)

# Calcul des embeddings
embeddings = compute_embeddings(words)

# Affichage des informations
display_model(model)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MODÈLE D'EMBEDDINGS

Architecture :
  SentenceTransformer

Modèle :
  sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2

Dimension des embeddings :
  384



In [40]:
#=================================================
# Cellule 5 - Projection
#=================================================

from explorer.display import display_projection
from explorer.projections import compute_pca

xyz, pca = compute_pca(embeddings)

explained_variance = pca.explained_variance_ratio_.sum() * 100
display_projection(explained_variance)

PROJECTION 3D

La projection 3D conserve environ 93.3% de la variance des embeddings.



In [41]:
#=================================================
# Cellule 6 - Dataframe
#=================================================

from explorer.dataframe import create_dataframe

df = create_dataframe(
    words,
    categories,
    xyz,
)

In [42]:
#================================================
# Cellule 7 - Similarities
#================================================

from explorer.similarities import compute_similarity
from explorer.similarities import rank_similarity_pairs
from explorer.display import display_similarity_ranking


similarities = compute_similarity(embeddings)

similarity_pairs = rank_similarity_pairs(words, similarities)

animal_icons = {
    "Chat": "🐱",
    "Lion": "🦁",
    "Loup": "🐺",
    "Chien": "🐕",
    "Renard": "🦊",
    "Tigre": "🐯",
}

display_similarity_ranking(similarity_pairs, animal_icons)

SEMANTIC SIMILARITY RANKING

 1. 🐕 Chien         ↔ 🐺 Loup          █████████████████              0.60
 2. 🐱 Chat          ↔ 🐺 Loup          ███████████████                0.52
 3. 🐺 Loup          ↔ 🦊 Renard        ██████████████                 0.49
 4. 🦁 Lion          ↔ 🐺 Loup          ████████████                   0.43
 5. 🐱 Chat          ↔ 🐕 Chien         ████████████                   0.41
 6. 🐕 Chien         ↔ 🦊 Renard        ████████████                   0.41
 7. 🦁 Lion          ↔ 🦊 Renard        ███████                        0.25
 8. 🐱 Chat          ↔ 🦊 Renard        ███████                        0.24
 9. 🦁 Lion          ↔ 🐕 Chien         █████                          0.17
10. 🐱 Chat          ↔ 🦁 Lion          ███                            0.13

------------------------------------------------------------
SUMMARY
------------------------------------------------------------

Most similar : 🐕 Chien ↔ 🐺 Loup (0.60)

Least similar: 🐱 Chat ↔ 🦁 Lion (0.13)



In [43]:
#==============================================
# Cellule 8 - Visualisation
#================================================

from explorer.display import display_model
from explorer.plotting import plot_scene
from explorer.plotting import CATEGORY_COLORS


fig = plot_scene(df, SCENARIO_NAME)
fig.show()

In [44]:
#======================================================================
# Cellule 9 Display exploration report
#======================================================================

from explorer.report import display_report

display_report(
    scenario_name=SCENARIO_NAME,
    model_name=MODEL_NAME,
    words=words,
    embedding_dimension=embeddings.shape[1],
    pca=pca,
    similarity_pairs=similarity_pairs,
)


LLM VISUAL EXPLORER REPORT

Scenario              : animals
Model                 : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Number of concepts    : 5
Embedding dimension   : 384

------------------------------------------------------------
3D PROJECTION (PCA)
------------------------------------------------------------
PC1 :  45.7%
PC2 :  29.2%
PC3 :  18.4%

384D → 3D preserves 93.3% of the semantic structure

------------------------------------------------------------
SEMANTIC OBSERVATIONS
------------------------------------------------------------
Closest concepts : Chien ↔ Loup (0.60)
Farthest concepts: Chat ↔ Lion (0.13)



In [45]:
#======================================================
# Pipeline Validation
#======================================================

print("=" * 60)
print("ÉTAT DU PIPELINE")
print("=" * 60)

print(f"Scénario          : {scenario['title']}")
print(f"Nombre d'objets   : {len(words)}")
print(f"Catégories        : {sorted(set(categories))}")
print(f"Modèle            : {MODEL_NAME}")
print(f"Embeddings        : {embeddings.shape}")
print(f"Projection        : {xyz.shape}")

print("=" * 60)

ÉTAT DU PIPELINE
Scénario          : Animaux
Nombre d'objets   : 5
Catégories        : ['Canidé', 'Félin']
Modèle            : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Embeddings        : (5, 384)
Projection        : (5, 3)


In [34]:
#======================================================
# Git commands (execute manually before release)
#======================================================
'''
git status
git diff --stat
git add .
git commit -m "LLM Visual Explorer V0.9.1 - stable release"
git tag v0.9.1
git push
git push --tags
'''

'\ngit status\ngit diff --stat\ngit add .\ngit commit -m "LLM Visual Explorer V0.9.1 - stable release"\ngit tag v0.9.1\ngit push\ngit push --tags\n'